## Brain Map Visualization - Cortical Thickness

### Overview

This notebook visualizes parcelwise **effect sizes (Hedges' g)** of cortical thickness
differences between groups, projected onto the **Desikan-Killiany (dk)** atlas using the
`ggseg` R package (Mowinckel & Vidal-Piñeiro, 2020).

**Effect size metric:** Hedges' g - a bias-corrected standardized mean difference.
Positive values = first group > second group; negative = first < second.
For PD vs HC contrasts, negative g ≈ atrophy (thinner cortex in PD).

**Why Hedges' g over T-statistic?** T-statistics conflate effect size with sample size;
large T can arise from trivial differences in large samples. Hedges' g is
sample-size-independent and directly comparable across contrasts and modalities.
The small-sample correction (J factor) reduces bias vs Cohen's d.

**Atlas:** Desikan-Killiany (`dk`), 34 regions per hemisphere = 68 parcels total.

**Contrasts:** De Novo PD vs HC · Prodromal PD vs HC · De Novo PD vs Prodromal PD

**Pipeline overview:**
1. Load aligned subject-level data (`df1_id_thick_aligned.csv`)
2. Compute Welch's t-test + Hedges' g per parcel per contrast
3. Apply FDR correction (Benjamini-Hochberg) within each contrast
4. Plot full Hedges' g map per contrast (ggseg dk atlas)
5. Plot FDR-masked map (non-significant parcels → grey)
6. Save figures and print results table

**ggseg API note (v2.x):** Atlases are functions `dk()` / `aseg()`; joins are on a
single `label` column (`lh_bankssts`, `rh_fusiform`, …). Only `label` + the fill
variable should be passed to `ggplot()` - extra columns with names overlapping the atlas
(e.g. `hemi`, `region`) break the automatic join.

**Reference:** Mowinckel & Vidal-Piñeiro (2020). Visualization of Brain Statistics With R
Packages ggseg and ggseg3d. *Advances in Methods and Practices in Psychological Science*.

In [ ]:
# ── Install packages if not already available ─────────────────────────────────
# Uncomment on first run:
# install.packages(c("tidyverse", "patchwork", "remotes"))
# remotes::install_github("LCBC-UiO/ggseg")

suppressPackageStartupMessages({
  library(ggseg)
  library(tidyverse)
  library(patchwork)
})

cat("ggseg version:", as.character(packageVersion("ggseg")), "\n")
cat("ggplot2 version:", as.character(packageVersion("ggplot2")), "\n")

### 1. Load pre-computed parcelwise OLS T-statistics

Three CSV files from the main analysis (`thickness_cortical_shi.ipynb`), one per contrast.
Each file contains 68 rows (34 LH + 34 RH parcels) with columns:
- `parcel` — NiSpace parcel ID, e.g. `hemi-L_lab-bankssts`
- `Tvalue` — OLS T-statistic (sign convention: T > 0 means g2/reference > g1/first-named)
- `pvalue` — two-tailed p-value
- `df` — residual degrees of freedom
- `hemi` — hemisphere code (`L` / `R`)

The OLS model included **age, sex, and field strength** as covariates.

In [ ]:
RESULTS_DIR <- "../../results"
FIG_DIR     <- file.path(RESULTS_DIR, "figures", "ggseg_thickness_cortical")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

GROUP_N <- list(
  "De Novo PD vs HC"           = c(n1 = 558, n2 = 192),
  "Prodromal PD vs HC"         = c(n1 = 279, n2 = 192),
  "De Novo PD vs Prodromal PD" = c(n1 = 558, n2 = 279)
)

TTEST_FILES <- list(
  "De Novo PD vs HC"           = file.path(RESULTS_DIR, "de novo pd vs hc_parcelwise_ttest.csv"),
  "Prodromal PD vs HC"         = file.path(RESULTS_DIR, "prodromal pd vs hc_parcelwise_ttest.csv"),
  "De Novo PD vs Prodromal PD" = file.path(RESULTS_DIR, "de novo pd vs prodromal pd_parcelwise_ttest.csv")
)

df_thick <- imap_dfr(TTEST_FILES, function(path, cname) {
  read_csv(path, show_col_types = FALSE) %>% mutate(contrast = cname)
}) %>%
  filter(!is.na(pvalue))

cat("Loaded:", nrow(df_thick), "parcel-contrast rows\n")
cat("Contrasts:", paste(unique(df_thick$contrast), collapse = " | "), "\n")

### 2. Derive Hedges' g from OLS T-statistics

For each of the 68 parcels and 3 contrasts we convert the OLS T-statistic to Hedges' g:

$$g = -T \cdot \sqrt{\frac{1}{n_1} + \frac{1}{n_2}} \cdot J(df)$$

$$J(df) = 1 - \frac{3}{4 \cdot df - 1}$$

**Sign convention:** The NiSpace OLS codes the group variable as `{first-named: 0, reference: 1}`,
so a positive T means reference > first-named. Negating T gives **g > 0 = first-named group
has greater thickness** (e.g. for "De Novo PD vs HC", g > 0 means PD thicker than HC).

For our sample sizes (n > 100 per group), J ≈ 0.999, making g ≈ Cohen's d.

**Why OLS T-stats instead of Welch's t-test?**
The OLS model includes age, sex, and field strength as covariates. Field strength (1.5T vs 3T)
systematically affects apparent cortical thickness — 3T scanners produce higher contrast images
that can inflate measured thickness values. Correcting for field strength in OLS removes this
confound. Running a simple Welch's t-test on raw (unadjusted) parcel values would not account
for scanner-driven group differences.

In [ ]:
# Named list kept for downstream visualization cells that loop over names(contrasts)
contrasts <- setNames(as.list(names(GROUP_N)), names(GROUP_N))

hedges_g_from_t <- function(t, n1, n2, df_resid) {
  # Sign negation: OLS T encodes (reference - first-named), so -T gives first-named - reference.
  # g > 0 means first-named group (e.g. De Novo PD) has greater thickness.
  -t * sqrt(1/n1 + 1/n2) * (1 - 3 / (4 * df_resid - 1))
}

df_stats <- df_thick %>%
  rowwise() %>%
  mutate(g = hedges_g_from_t(
    Tvalue,
    GROUP_N[[contrast]][["n1"]],
    GROUP_N[[contrast]][["n2"]],
    df
  )) %>%
  ungroup() %>%
  mutate(
    hemi   = if_else(hemi == "L", "left", "right"),
    region = str_replace(parcel, ".*_lab-", ""),
    label  = paste0(if_else(hemi == "left", "lh", "rh"), "_", region)
  )

cat("Stats computed for", nrow(df_stats), "parcel × contrast combinations\n")
cat("Hedges' g range:", round(range(df_stats$g, na.rm = TRUE), 3), "\n")

### 3. FDR correction

Benjamini-Hochberg FDR correction applied **within each contrast** (68 tests per contrast).
Regions with `p_fdr < 0.05` are shown coloured in the masked maps; all others are grey.

In [ ]:
df_stats <- df_stats %>%
  group_by(contrast) %>%
  mutate(p_fdr = p.adjust(pvalue, method = "fdr")) %>%
  ungroup()

df_stats %>%
  group_by(contrast) %>%
  summarise(
    n_sig_fdr   = sum(p_fdr < 0.05, na.rm = TRUE),
    n_sig_uncor = sum(pvalue < 0.05, na.rm = TRUE),
    n_total     = n(),
    g_max_abs   = round(max(abs(g), na.rm = TRUE), 3)
  )

### 4. Colour scale

Diverging blue–white–red scale centred at 0 (no difference):
- **Blue (#2166AC)**: negative g - first group has *thinner* cortex
- **Red (#D6604D)**: positive g - first group has *thicker* cortex
- **Grey (grey85)**: no data or below FDR threshold

Scale limits are ±0.5 (medium effect size); values beyond are squished to the boundary
colour. Adjust `G_LIMIT` if effects are consistently larger or smaller.

In [ ]:
G_LIMIT <- 0.5   # |g| = 0.5 corresponds to a medium-to-large effect

scale_g <- scale_fill_gradient2(
  low      = "#2166AC",
  mid      = "white",
  high     = "#D6604D",
  midpoint = 0,
  limits   = c(-G_LIMIT, G_LIMIT),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

### 5. Brain maps per contrast

For each contrast we display a pair of maps side by side:
- **Left panel**: full Hedges' g map (all 68 parcels coloured)
- **Right panel**: FDR-masked map (only parcels with q < 0.05 coloured; rest grey)

**ggseg 2.x usage pattern** (following Mowinckel & Vidal-Piñeiro, 2020):
```r
ggplot(data %>% select(label, value)) +
  geom_brain(atlas = dk(), mapping = aes(fill = value), ...)
```
Only `label` + the fill variable are passed to `ggplot()`. Any extra column that shares
a name with an atlas column (`hemi`, `region`, etc.) will be incorrectly used as an
additional join key and break the merge.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

# dk_4view: lateral + medial only (defined in section 5c; run that cell first)
# Fallback: build it here if this cell is run independently
if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

for (cname in names(contrasts)) {

  p_full <- ggplot(df_stats %>% filter(contrast == cname) %>% select(label, g)) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g), colour = "white",
               position = position_brain("horizontal")) +
    scale_g +
    labs(title = cname, subtitle = "Cortical thickness - Hedges' g (all parcels)") +
    theme_brain2() +
    theme(plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  p_masked <- ggplot(
    df_stats %>% filter(contrast == cname) %>%
      mutate(g_sig = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_sig)
  ) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_sig), colour = "white",
               position = position_brain("horizontal")) +
    scale_g +
    labs(title = cname, subtitle = "Cortical thickness - FDR-masked (q < 0.05)") +
    theme_brain2() +
    theme(plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  combined <- p_full + p_masked + plot_layout(guides = "collect") & theme(legend.position = "right")
  print(combined)

  fname <- tolower(str_replace_all(cname, " ", "_"))
  ggsave(file.path(FIG_DIR, paste0(fname, ".png")), combined, width = 14, height = 5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 5b. Paper-style brain maps (sequential red scale, FDR-only)

Replicating the style of Laansma et al. (following the figure caption):
> *"Cohen's d values were calculated and are presented in the figure when the FDR-adjusted
> p value reached < 0.05. Darker red indicates more atrophy."*

**Design choices:**
- **Sequential white → dark red** scale: encodes *magnitude* of atrophy, not direction
- **Absolute |Hedges' g|** displayed — only for FDR-significant parcels
- **Non-significant parcels** → very light grey (`#f5f5f5`) so the brain outline stays visible
- **Grey parcel borders** (`grey60`) make individual parcels distinguishable
- One panel per contrast (not paired full + masked)

In [ ]:
FIG_DIR_PAPER <- file.path(FIG_DIR, "paper_style")
dir.create(FIG_DIR_PAPER, recursive = TRUE, showWarnings = FALSE)

G_LIMIT_PAPER <- ceiling(max(abs(df_stats$g), na.rm = TRUE) * 10) / 10

scale_g_paper <- scale_fill_gradient(
  low      = "white",
  high     = "#B2182B",
  limits   = c(0, G_LIMIT_PAPER),
  oob      = scales::squish,
  name     = "|Hedges' g|",
  na.value = "#f5f5f5"
)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

cat("Paper scale upper limit:", G_LIMIT_PAPER, "\n")
options(repr.plot.width = 10, repr.plot.height = 4)

for (cname in names(contrasts)) {
  plot_data <- df_stats %>%
    filter(contrast == cname) %>%
    mutate(g_abs = if_else(p_fdr < 0.05, abs(g), NA_real_)) %>%
    select(label, g_abs)

  p_paper <- ggplot(plot_data) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_abs), colour = "grey60",
               position = position_brain("horizontal")) +
    scale_g_paper +
    labs(title = cname, subtitle = "Cortical thickness - |Hedges' g| (FDR q < 0.05 only)") +
    theme_brain2() +
    theme(plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  print(p_paper)

  fname <- paste0("paper_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(file.path(FIG_DIR_PAPER, paste0(fname, ".png")), p_paper, width = 10, height = 4, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 5c. Cortical thinning maps: 4-view layout (lateral + medial only)

Two publication-ready combined figures, each showing all 3 contrasts stacked vertically.
Only **lateral** and **medial** views are rendered (inferior/superior views removed),
giving one row of 4 brain images per contrast (LH lateral · LH medial · RH medial · RH lateral).

**Figure 5c-1 — Unthresholded thinning:** Shows all parcels where g < 0 (first-named group
thinner), regardless of significance. Communicates the full sub-threshold atrophy landscape
that motivates colocalization analysis. Panel labels: **a / b / c** (one per contrast).

**Figure 5c-2 — FDR-corrected thinning:** Only parcels with g < 0 **and** q < 0.05.
Because no cortical thickness parcel survives FDR correction, this figure is intentionally
blank — making the sub-threshold nature of the findings visually explicit. Panel labels: **a / b / c**.

**Colour scale:** white → pink, auto-scaled to the maximum observed thinning effect.

In [ ]:
FIG_DIR_ATROPHY <- file.path(FIG_DIR, "atrophy_style")
dir.create(FIG_DIR_ATROPHY, recursive = TRUE, showWarnings = FALSE)

# Build 4-view atlas: keep only lateral + medial (removes inferior/superior)
dk_4view <- dk()
dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))

max_atrophy <- ceiling(max(abs(df_stats$g[df_stats$g < 0]), na.rm = TRUE) * 20) / 20
cat("Atrophy scale upper limit:", max_atrophy, "\n")

scale_atrophy <- scale_fill_gradient(
  low      = "white",
  high     = "pink",
  limits   = c(0, max_atrophy),
  oob      = scales::squish,
  na.value = "grey90",
  name     = "Atrophy\n(|Hedges' g|)"
)

make_brain_panel <- function(cname, fdr_only = FALSE) {
  d <- df_stats %>% filter(contrast == cname)
  n_sig <- sum(d$p_fdr < 0.05 & d$g < 0, na.rm = TRUE)
  if (fdr_only) {
    plot_data <- d %>%
      mutate(atrophy = if_else(g < 0 & p_fdr < 0.05, abs(g), NA_real_)) %>%
      select(label, atrophy)
    subt <- if (n_sig == 0) "No parcel survives FDR correction (q < 0.05)" else
      paste0("FDR-significant thinning: n = ", n_sig, " parcels (q < 0.05)")
  } else {
    plot_data <- d %>%
      mutate(atrophy = if_else(g < 0, abs(g), NA_real_)) %>%
      select(label, atrophy)
    n_unc <- sum(d$pvalue < 0.05 & d$g < 0, na.rm = TRUE)
    subt <- paste0(n_unc, " parcels p < 0.05 uncorrected (all thinning)")
  }
  ggplot(plot_data) +
    geom_brain(
      atlas    = dk_4view,
      mapping  = aes(fill = atrophy),
      colour   = "grey70",
      position = position_brain("horizontal")
    ) +
    scale_atrophy +
    labs(title = cname, subtitle = subt) +
    theme_void() +
    theme(
      plot.title    = element_text(hjust = 0.5, size = 11, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 8.5, colour = "grey35"),
      legend.position = "right"
    )
}

# ── Figure 5c-1: Unthresholded thinning (a / b / c) ───────────────────────
options(repr.plot.width = 10, repr.plot.height = 12)

panels_unc <- map(names(contrasts), ~ make_brain_panel(.x, fdr_only = FALSE))

fig_unc <- wrap_plots(panels_unc, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Thickness: Sub-threshold Thinning Across the Disease Continuum",
    subtitle = "Pink = cortical thinning (g < 0), |Hedges' g|, unthresholded. Grey = no thinning. No parcel survives FDR correction.",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_unc)
ggsave(
  file.path(FIG_DIR_ATROPHY, "figure5c1_cortical_thickness_unthresholded.png"),
  fig_unc, width = 10, height = 12, dpi = 300
)
cat("Saved: figure5c1_cortical_thickness_unthresholded.png\n")

# ── Figure 5c-2: FDR-corrected thinning (a / b / c) ───────────────────────
panels_fdr <- map(names(contrasts), ~ make_brain_panel(.x, fdr_only = TRUE))

fig_fdr <- wrap_plots(panels_fdr, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Thickness: FDR-corrected Thinning (q < 0.05)",
    subtitle = "FDR-corrected thinning only (g < 0, q < 0.05). No parcel survives FDR correction in any contrast - all panels appear grey.",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_fdr)
ggsave(
  file.path(FIG_DIR_ATROPHY, "figure5c2_cortical_thickness_fdr_corrected.png"),
  fig_fdr, width = 10, height = 12, dpi = 300
)
cat("Saved: figure5c2_cortical_thickness_fdr_corrected.png\n")

### Figure 5c — Thesis caption

**Figure 5c-1 (unthresholded).** Cortical thickness atrophy maps across three group contrasts: (a) De Novo PD vs HC, (b) Prodromal PD vs HC, (c) De Novo PD vs Prodromal PD. Pink shading indicates parcels where the first-named group has reduced cortical thickness relative to the reference group (Hedges' g < 0); colour intensity encodes effect magnitude (|Hedges' g|). Grey parcels show no thinning or greater thickness in the first-named group. Maps display lateral and medial views of both hemispheres. No parcel survives FDR correction (Benjamini-Hochberg q < 0.05) in any contrast; the number of uncorrected nominally significant parcels (p < 0.05) per contrast is shown in each panel subtitle. OLS T-statistics adjusted for age, sex, and MRI field strength.

**Figure 5c-2 (FDR-corrected).** Same contrasts as 5c-1, retaining only parcels that survive FDR correction (q < 0.05). All panels are uniformly grey, confirming the absence of FDR-significant cortical thinning across all contrasts. The sub-threshold, spatially distributed thinning visible in Figure 5c-1 motivates the neurotransmitter colocalization analyses in Section 3.3.

### 5d. Combined atrophy figure: unthresholded vs FDR-corrected (labelled a–f)

Publication-ready 3 × 2 figure combining all contrasts and both thresholding approaches
in a single panel for direct comparison.

**Layout:**

| | Left column | Right column |
|---|---|---|
| **Row 1 (a, b)** | De Novo PD vs HC — all thinning | De Novo PD vs HC — FDR q < 0.05 |
| **Row 2 (c, d)** | Prodromal PD vs HC — all thinning | Prodromal PD vs HC — FDR q < 0.05 |
| **Row 3 (e, f)** | De Novo PD vs Prodromal PD — all thinning | De Novo PD vs Prodromal PD — FDR q < 0.05 |

**Left panels (a, c, e):** All parcels with g < 0 (cortical thinning), regardless of
significance. Communicates the *spatial landscape* that motivates colocalization analyses.

**Right panels (b, d, f):** Only FDR-significant thinning (g < 0 and q < 0.05). Because
no cortical thickness parcel survives FDR correction, these panels appear grey — the
contrast with the left panels makes the sub-threshold nature of the findings explicit.

Scale: white → pink, |Hedges' g|, limits 0–0.30.
Panel letters (a–f) added automatically via patchwork `tag_levels`.

In [ ]:
FIG_DIR_COMBINED <- file.path(FIG_DIR, "combined_atrophy")
dir.create(FIG_DIR_COMBINED, recursive = TRUE, showWarnings = FALSE)

scale_pink <- scale_fill_gradient(
  low      = "white",
  high     = "pink",
  limits   = c(0, max_atrophy),
  oob      = scales::squish,
  na.value = "grey90",
  name     = "|Hedges' g|\natrophy"
)

contrast_titles <- c(
  "De Novo PD vs HC"           = "De Novo PD vs HC",
  "Prodromal PD vs HC"         = "Prodromal PD vs HC",
  "De Novo PD vs Prodromal PD" = "De Novo PD vs Prodromal PD"
)

plots_unthresh <- list()
plots_fdr_only <- list()

for (cname in names(contrasts)) {
  d     <- df_stats %>% filter(contrast == cname)
  n_sig <- sum(d$p_fdr < 0.05 & d$g < 0, na.rm = TRUE)

  plots_unthresh[[cname]] <- ggplot(
    d %>% mutate(atrophy = if_else(g < 0, abs(g), NA_real_)) %>% select(label, atrophy)
  ) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = atrophy), colour = "grey70",
               position = position_brain("horizontal")) +
    scale_pink +
    labs(title = contrast_titles[[cname]], subtitle = "All cortical thinning (g < 0, unthresholded)") +
    theme_void() +
    theme(plot.title    = element_text(hjust = 0.5, size = 10, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 8.5, colour = "grey35"))

  plots_fdr_only[[cname]] <- ggplot(
    d %>% mutate(atrophy = if_else(g < 0 & p_fdr < 0.05, abs(g), NA_real_)) %>% select(label, atrophy)
  ) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = atrophy), colour = "grey70",
               position = position_brain("horizontal")) +
    scale_pink +
    labs(title = contrast_titles[[cname]],
         subtitle = paste0("FDR-corrected thinning (q < 0.05; n = ", n_sig, " parcels)")) +
    theme_void() +
    theme(plot.title    = element_text(hjust = 0.5, size = 10, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 8.5, colour = "grey35"))
}

fig_combined <- (
  (plots_unthresh[[1]] | plots_fdr_only[[1]]) /
  (plots_unthresh[[2]] | plots_fdr_only[[2]]) /
  (plots_unthresh[[3]] | plots_fdr_only[[3]])
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title      = "Cortical Thickness: Unthresholded vs FDR-corrected Atrophy Maps",
    subtitle   = "Left (a, c, e): all thinning (g < 0) | Right (b, d, f): FDR-corrected (q < 0.05)",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

options(repr.plot.width = 14, repr.plot.height = 12)
print(fig_combined)

out_path <- file.path(FIG_DIR_COMBINED, "combined_cortical_thinning_unthresh_vs_fdr.png")
ggsave(out_path, fig_combined, width = 14, height = 12, dpi = 300)
cat("Saved:", out_path, "\n")

### 5e. Diverging pastel maps: both directions (light pink / light blue)

Publication-ready figure showing **both positive and negative Hedges' g simultaneously**
using a pastel diverging colour scale that matches the study's existing palette:

- **Light pink** (`lightpink`): negative g — first-named group has *thinner* cortex
- **Light blue** (`#AED6F1`): positive g — first-named group has *thicker* cortex
- **White**: no difference (g = 0)
- **Grey85**: no data

Unlike the atrophy-only (section 5c) figures which show only thinning (g < 0) in pink,
this figure shows the full landscape of cortical thickness differences in both directions,
making incidental hypertrophy (e.g. right parahippocampal in De Novo PD vs HC, g = +0.207)
directly visible alongside thinning regions.

Scale limits: ±0.30, which encompasses the full observed range (|g|_max ≈ 0.285).
All 68 parcels are coloured; no significance threshold applied.

In [ ]:
CNAMES_DISPLAY <- c(
  "De Novo PD vs HC"           = "de novo PD vs HC",
  "Prodromal PD vs HC"         = "prodromal PD vs HC",
  "De Novo PD vs Prodromal PD" = "de novo PD vs prodromal PD"
)

G_LIMIT_PASTEL <- 0.30

scale_diverging_pastel <- scale_fill_gradient2(
  low      = "lightpink",
  mid      = "white",
  high     = "#AED6F1",
  midpoint = 0,
  limits   = c(-G_LIMIT_PASTEL, G_LIMIT_PASTEL),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

make_pastel_thick <- function(cname, fdr_only = FALSE) {
  d <- df_stats %>% filter(contrast == cname)
  if (fdr_only) {
    plot_data <- d %>% mutate(g_plot = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_plot)
  } else {
    plot_data <- d %>% mutate(g_plot = g) %>% select(label, g_plot)
  }
  ggplot(plot_data) +
    geom_brain(
      atlas    = dk_4view,
      mapping  = aes(fill = g_plot),
      colour   = "grey70",
      position = position_brain("horizontal")
    ) +
    scale_diverging_pastel +
    labs(title = CNAMES_DISPLAY[[cname]]) +
    theme_void() +
    theme(
      plot.title    = element_text(hjust = 0.5, size = 11, face = "bold",
                                   margin = margin(t = 10, b = 3)),
      legend.position = "right",
      plot.margin   = margin(t = 8, r = 8, b = 12, l = 8)
    )
}

# Figure 5e-1: unthresholded (all 68 parcels)
options(repr.plot.width = 10, repr.plot.height = 12)

panels_unc <- map(names(contrasts), ~make_pastel_thick(.x, fdr_only = FALSE))

fig_diverging_unc <- wrap_plots(panels_unc, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Thickness: Diverging Hedges' g (unthresholded)",
    subtitle = "Light pink = thinning (g < 0) | Light blue = thickening (g > 0) | Scale +/-0.30",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_diverging_unc)

out_unc <- file.path(FIG_DIR, "figure_diverging_cortical_thickness_unthresh.png")
ggsave(out_unc, fig_diverging_unc, width = 10, height = 12, dpi = 300)
cat("Saved:", out_unc, "\n")

# Figure 5e-2: FDR-masked (only FDR-significant parcels colored)
panels_fdr <- map(names(contrasts), ~make_pastel_thick(.x, fdr_only = TRUE))

fig_diverging_fdr <- wrap_plots(panels_fdr, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Thickness: Diverging Hedges' g (FDR q < 0.05)",
    subtitle = "Light pink = thinning (g < 0) | Light blue = thickening (g > 0) | Grey = not significant",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_diverging_fdr)

out_fdr <- file.path(FIG_DIR, "figure_diverging_cortical_thickness_fdr.png")
ggsave(out_fdr, fig_diverging_fdr, width = 10, height = 12, dpi = 300)
cat("Saved:", out_fdr, "\n")

### Figure 5e — Thesis caption

**Figure 5e-1 (unthresholded).** Diverging cortical thickness maps across three group contrasts: (a) *de novo* PD vs HC, (b) prodromal PD vs HC, (c) *de novo* PD vs prodromal PD. Light pink parcels indicate cortical thinning in the first-named group (Hedges' g < 0); light blue indicates greater thickness (g > 0). Colour intensity encodes effect magnitude; white indicates no difference (g = 0). All 68 parcels shown regardless of significance threshold. Panel subtitles report the number of nominally significant parcels (p < 0.05 uncorrected) in each direction. No parcel survives FDR correction in any contrast.

**Figure 5e-2 (FDR-masked).** Same contrasts as 5e-1, showing only parcels that survive FDR correction (Benjamini–Hochberg q < 0.05) in either direction. All panels appear uniformly grey, confirming the absence of FDR-significant cortical thickness differences across all contrasts. The sub-threshold, spatially distributed effects visible in Figure 5e-1 motivate the neurotransmitter colocalization analyses.

### 6. Results table

All parcels sorted by contrast and absolute Hedges' g. FDR-significant parcels (q < 0.05)
are indicated with `*`.

In [ ]:
df_stats %>%
  select(contrast, hemi, region, g, Tvalue, pvalue, p_fdr) %>%
  mutate(
    across(where(is.numeric), \(x) round(x, 4)),
    sig = if_else(p_fdr < 0.05, "*", "")
  ) %>%
  arrange(contrast, desc(abs(g)))

---
## Notes

### Effect size: Hedges' g
Hedges' g is the preferred effect size metric for group comparisons because it:
(1) is independent of sample size (unlike T), (2) corrects for small-sample positive bias
via the J factor, and (3) is directly interpretable: |g| ≈ 0.2 small, 0.5 medium, 0.8 large.

### Direction
g > 0 → first group named in contrast has *greater* thickness.
g < 0 → first group has *less* thickness (atrophy). For "De Novo PD vs HC", g < 0 = PD thinner.

### Data source
Effect sizes are derived from OLS T-statistics saved in `results/*_parcelwise_ttest.csv`
by the main analysis notebook (`thickness_cortical_shi.ipynb`). The OLS model included
**age, sex, and field strength** as covariates — these are NOT in the raw aligned CSV,
which is why this notebook loads pre-computed T-stats rather than raw subject-level data.

### ggseg 2.x API
- Atlases are **functions**: `dk()`, `aseg()` (not data objects as in ggseg 1.x)
- Join key: single `label` column (`"lh_bankssts"`, `"rh_fusiform"`, …)
- Only pass `label` + fill variable to `ggplot()` - extra columns cause incorrect joins
- `position_brain("stacked")` is broken in v2.1.0; use `"horizontal"` instead

**Group codes:** CONCOHORT 1 = De Novo PD · 2 = HC · 4 = Prodromal PD